# Training Mechanics — Hands-On Notebook

Companion to: [Backpropagation](../docs/00_prerequisites/03_backpropagation.md), [Loss & Regularization](../docs/00_prerequisites/04_loss_and_regularization.md)

**What you'll build:**
- Manual forward and backward pass with real numbers
- SGD, Momentum, and Adam from scratch
- Train a classifier on MNIST comparing optimizers
- See dropout and L2 regularization in action

In [ ]:
# Section 1: Setup
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)

## Section 2: Manual Forward & Backward Pass

Let's trace a complete learning step by hand on a tiny network: **2 inputs → 2 hidden (ReLU) → 1 output (sigmoid)**, with **MSE** loss. We print every intermediate value, then derive gradients with the chain rule and check them against PyTorch.

In [ ]:
# Forward: x is 1x2, shapes chosen so x @ W1 + b1 works
x = np.array([[1.0, 2.0]], dtype=np.float64)
y_true = np.array([[0.8]], dtype=np.float64)

W1 = np.array([[0.1, 0.3], [0.2, 0.4]], dtype=np.float64)  # 2x2
b1 = np.array([[0.01, 0.02]], dtype=np.float64)
W2 = np.array([[0.5], [0.6]], dtype=np.float64)  # 2x1
b2 = np.array([[0.1]], dtype=np.float64)

z1 = x @ W1 + b1
print("z1 (pre-activation hidden):", z1)
a1 = np.maximum(z1, 0.0)
print("a1 (ReLU):", a1)
z2 = a1 @ W2 + b2
print("z2 (logit):", z2)
y_hat = 1.0 / (1.0 + np.exp(-z2))
print("y_hat (sigmoid):", y_hat)
loss = 0.5 * np.mean((y_hat - y_true) ** 2)
print("MSE loss (scalar, 1 sample):", float(loss))

# Backward: L = 1/2 * (y_hat - y)^2  =>  dL/dy_hat = y_hat - y
dL_dy_hat = y_hat - y_true
sigmoid_prime = y_hat * (1.0 - y_hat)
dL_dz2 = dL_dy_hat * sigmoid_prime
print("\nGradients (chain rule):")
print("dL/dz2:", dL_dz2)
dL_dW2 = a1.T @ dL_dz2
dL_db2 = dL_dz2
print("dL/dW2:\n", dL_dW2)
print("dL/db2:", dL_db2)
dL_da1 = dL_dz2 @ W2.T
relu_mask = (z1 > 0).astype(np.float64)
dL_dz1 = dL_da1 * relu_mask
print("dL/dz1:", dL_dz1)
dL_dW1 = x.T @ dL_dz1
dL_db1 = dL_dz1
print("dL/dW1:\n", dL_dW1)
print("dL/db1:", dL_db1)

In [ ]:
# Same tensors in PyTorch — autograd should match manual grads
x_t = torch.tensor(x, dtype=torch.float64)
y_t = torch.tensor(y_true, dtype=torch.float64)
W1_t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2_t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

z1_t = x_t @ W1_t + b1_t
a1_t = torch.relu(z1_t)
z2_t = a1_t @ W2_t + b2_t
y_hat_t = torch.sigmoid(z2_t)
loss_t = 0.5 * torch.mean((y_hat_t - y_t) ** 2)
loss_t.backward()

print("PyTorch loss:", loss_t.item())
print("|dW1| max abs diff:", (W1_t.grad.numpy() - dL_dW1).max())
print("|db1| max abs diff:", (b1_t.grad.numpy() - dL_db1).max())
print("|dW2| max abs diff:", (W2_t.grad.numpy() - dL_dW2).max())
print("|db2| max abs diff:", (b2_t.grad.numpy() - dL_db2).max())

## Section 3: Loss Functions Side-by-Side

- **MSE** \((y - \hat{y})^2\): good for regression; for probabilities it can be slow to penalize confident wrong answers.
- **Binary cross-entropy (BCE)** \(-[y\log\hat{y} + (1-y)\log(1-\hat{y})]\): standard for a single binary label in \([0,1]\).
- **Multi-class cross-entropy** \(-\sum_k y_k \log \hat{y}_k\): use with **softmax** outputs; the target is often a one-hot vector (or class indices with `CrossEntropyLoss`, which combines log-softmax and NLL).

Below we compute small numeric examples by hand and with `torch.nn`, then plot how MSE and BCE behave when the true label is **1** and the prediction **p** varies.

In [ ]:
# Small example: one sample, binary target y=1, prediction p=0.7
y_bin = torch.tensor([1.0])
p = torch.tensor([0.7], requires_grad=False)

mse_hand = (p - y_bin) ** 2
bce_hand = -(y_bin * torch.log(p) + (1 - y_bin) * torch.log(1 - p))

bce_torch = nn.BCELoss()(p, y_bin)
mse_torch = nn.MSELoss()(p, y_bin)
print("MSE hand:", float(mse_hand), "torch:", float(mse_torch))
print("BCE hand:", float(bce_hand), "torch:", float(bce_torch))

# Multi-class: true class 2 of 3, softmax probs
logits = torch.tensor([[1.0, 0.5, 2.0]])
probs = torch.softmax(logits, dim=1)
target_idx = torch.tensor([2])
ce_torch = nn.CrossEntropyLoss()(logits, target_idx)
log_probs = torch.log(probs + 1e-12)
ce_hand = -log_probs[0, 2]
print("Multi-class CE hand:", float(ce_hand), "torch:", float(ce_torch))

In [ ]:
ps = torch.linspace(0.01, 0.99, 200)
y_one = torch.ones_like(ps)
mse_curve = (ps - y_one) ** 2
bce_curve = -(y_one * torch.log(ps) + (1 - y_one) * torch.log(1 - ps))

plt.figure(figsize=(7, 4))
plt.plot(ps.numpy(), mse_curve.numpy(), label="MSE (y=1)")
plt.plot(ps.numpy(), bce_curve.numpy(), label="BCE (y=1)")
plt.xlabel("prediction p")
plt.ylabel("loss")
plt.title("MSE vs BCE when true label is 1")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Cross-entropy** penalizes confident **wrong** predictions much more harshly than MSE: when \(y=1\) but \(p\to 0\), BCE blows up (we clip \(p\) in practice), while MSE stays bounded at 1.

## Section 4: Optimizers from Scratch

**SGD** = take a step proportional to the gradient: \(\theta \leftarrow \theta - \eta \nabla L\).

In [ ]:
# Minimize f(x) = x^2, starting at x = 5.0
def grad_f(x):
    return 2 * x

x0 = 5.0
lr = 0.05
n_iter = 60

x_sgd = x0
traj_sgd = [x_sgd]
for _ in range(n_iter):
    x_sgd = x_sgd - lr * grad_f(x_sgd)
    traj_sgd.append(x_sgd)
traj_sgd = np.array(traj_sgd)
print("SGD final x:", traj_sgd[-1])

**Momentum** = SGD with a memory of past gradients: \(v \leftarrow \beta v + \nabla L\), then \(\theta \leftarrow \theta - \eta v\).

In [ ]:
beta = 0.9
x_mom = x0
v = 0.0
traj_mom = [x_mom]
for _ in range(n_iter):
    g = grad_f(x_mom)
    v = beta * v + g
    x_mom = x_mom - lr * v
    traj_mom.append(x_mom)
traj_mom = np.array(traj_mom)
print("Momentum final x:", traj_mom[-1])

**Adam** = adaptive learning rates + momentum-like estimates of first and second moments (implemented step-by-step for one scalar \(x\)).

In [ ]:
lr_adam = 0.5
beta1, beta2 = 0.9, 0.999
eps = 1e-8
x_adam = x0
m_t = 0.0
v_t = 0.0
traj_adam = [x_adam]
for t in range(1, n_iter + 1):
    g = grad_f(x_adam)
    m_t = beta1 * m_t + (1 - beta1) * g
    v_t = beta2 * v_t + (1 - beta2) * (g ** 2)
    m_hat = m_t / (1 - beta1 ** t)
    v_hat = v_t / (1 - beta2 ** t)
    x_adam = x_adam - lr_adam * m_hat / (np.sqrt(v_hat) + eps)
    traj_adam.append(x_adam)
traj_adam = np.array(traj_adam)
print("Adam final x:", traj_adam[-1])

In [ ]:
iters = np.arange(len(traj_sgd))
plt.figure(figsize=(7, 4))
plt.plot(iters, np.abs(traj_sgd), label="SGD |x|")
plt.plot(iters, np.abs(traj_mom), label="Momentum |x|")
plt.plot(iters, np.abs(traj_adam), label="Adam |x|")
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("|x| (log scale)")
plt.title("Convergence to minimum of f(x)=x² (x*=0)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Iterations to |x|<1e-3:")
for name, tr in [("SGD", traj_sgd), ("Momentum", traj_mom), ("Adam", traj_adam)]:
    idx = np.where(np.abs(tr) < 1e-3)[0]
    print(f"  {name}: {idx[0] if len(idx) else 'not reached'} ")

## Section 5: Learning Rate Schedules

In [ ]:
epochs = 100
t = np.arange(epochs, dtype=np.float64)
eta_max = 1e-2
eta_min = 1e-4
warmup = 10

warm = np.minimum(t, warmup) / warmup
cosine = eta_min + 0.5 * (eta_max - eta_min) * (1 + np.cos(np.pi * (t - warmup) / max(epochs - warmup, 1)))
warm_cos = np.where(t < warmup, eta_max * warm, cosine)

step_decay = eta_max * (0.5 ** (t // 30))
exp_decay = eta_max * np.exp(-0.03 * t)

plt.figure(figsize=(8, 4))
plt.plot(t, warm_cos, label="Warmup + cosine decay")
plt.plot(t, step_decay, label="Step decay (÷2 every 30 epochs)")
plt.plot(t, exp_decay, label="Exponential decay")
plt.xlabel("epoch")
plt.ylabel("learning rate")
plt.title("Example learning-rate schedules (100 epochs)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Warmup** reduces early instability when gradients are large; **decay** shrinks the step size so training can fine-tune in later epochs.

## Section 6: Training on MNIST — Optimizer Comparison

We use the first **10k** training images and a small **784 → 128 → 10** MLP for **10 epochs**.

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_train = datasets.MNIST(root="../data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="../data", train=False, download=True, transform=tfm)
n_sub = 10_000
idx = torch.randperm(len(full_train))[:n_sub]
train_ds = torch.utils.data.Subset(full_train, idx)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


def run_epoch(model, loader, opt, criterion, train=True):
    if train:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    n = 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb, yb
            if train:
                opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                loss.backward()
                opt.step()
            total_loss += loss.item() * xb.size(0)
            n += xb.size(0)
    return total_loss / n


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct, tot = 0, 0
    for xb, yb in loader:
        pred = model(xb).argmax(dim=1)
        correct += (pred == yb).sum().item()
        tot += yb.size(0)
    return correct / tot


def train_mnist(optimizer_ctor, name, epochs=10):
    torch.manual_seed(42)
    model = MLP()
    criterion = nn.CrossEntropyLoss()
    opt = optimizer_ctor(model.parameters())
    losses = []
    for ep in range(epochs):
        losses.append(run_epoch(model, train_loader, opt, criterion, train=True))
    acc = accuracy(model, test_loader)
    return losses, acc, name


epochs_m = 10
lr_m = 0.05
res_sgd = train_mnist(
    lambda p: optim.SGD(p, lr=lr_m), "SGD", epochs_m
)
res_mom = train_mnist(
    lambda p: optim.SGD(p, lr=lr_m, momentum=0.9), "SGD+Momentum", epochs_m
)
res_adam = train_mnist(
    lambda p: optim.Adam(p, lr=1e-3), "Adam", epochs_m
)

plt.figure(figsize=(7, 4))
for losses, acc, name in (res_sgd, res_mom, res_adam):
    plt.plot(range(1, epochs_m + 1), losses, label=f"{name}")
plt.xlabel("epoch")
plt.ylabel("average train loss")
plt.title("MNIST: training loss by optimizer")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Final test accuracy:")
for _, acc, name in (res_sgd, res_mom, res_adam):
    print(f"  {name}: {acc:.4f}")

## Section 7: Regularization in Action

**Regularization** reduces **overfitting**: when the model memorizes training noise instead of learning patterns that generalize.

In [ ]:
tiny_n = 200
tiny_idx = torch.randperm(len(full_train))[:tiny_n]
tiny_train = torch.utils.data.Subset(full_train, tiny_idx)
tiny_loader = DataLoader(tiny_train, batch_size=32, shuffle=True)


class BigMLP(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        layers = [
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.ReLU(),
        ]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        layers.extend([nn.Linear(512, 10)])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_tiny(model, opt, epochs=40, wd=0.0):
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in tiny_loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
    train_acc = accuracy(model, tiny_loader)
    test_acc = accuracy(model, test_loader)
    return train_acc, test_acc


torch.manual_seed(0)
m_none = BigMLP(dropout_p=0.0)
opt_none = optim.Adam(m_none.parameters(), lr=1e-3, weight_decay=0.0)
tr0, te0 = train_tiny(m_none, opt_none)
print(f"No reg — train acc: {tr0:.3f}, test acc: {te0:.3f}")

In [ ]:
torch.manual_seed(0)
m_l2 = BigMLP(dropout_p=0.0)
opt_l2 = optim.Adam(m_l2.parameters(), lr=1e-3, weight_decay=1e-3)
tr_l2, te_l2 = train_tiny(m_l2, opt_l2)
print(f"L2 (weight_decay) — train acc: {tr_l2:.3f}, test acc: {te_l2:.3f}")

In [ ]:
torch.manual_seed(0)
m_do = BigMLP(dropout_p=0.5)
opt_do = optim.Adam(m_do.parameters(), lr=1e-3, weight_decay=0.0)
tr_do, te_do = train_tiny(m_do, opt_do)
print(f"Dropout(0.5) — train acc: {tr_do:.3f}, test acc: {te_do:.3f}")

In [ ]:
torch.manual_seed(0)
m_both = BigMLP(dropout_p=0.5)
opt_both = optim.Adam(m_both.parameters(), lr=1e-3, weight_decay=1e-3)
tr_b, te_b = train_tiny(m_both, opt_both)
print(f"L2 + Dropout — train acc: {tr_b:.3f}, test acc: {te_b:.3f}")

labels = ["No reg", "L2 only", "Dropout only", "L2+Dropout"]
tests = [te0, te_l2, te_do, te_b]
plt.figure(figsize=(7, 4))
plt.bar(labels, tests, color=["#888", "#59f", "#f95", "#6c6"])
plt.ylabel("test accuracy")
plt.title("Tiny MNIST (200 samples): regularization vs test accuracy")
plt.ylim(0, 1)
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Section 8: Batch Normalization vs Layer Normalization

- **BatchNorm** normalizes each feature dimension using **statistics across the batch** (and running averages at eval time). Very common in CNNs with large batches.
- **LayerNorm** normalizes **across the feature dimension** for each example — batch size does not affect the normalization statistic.

In [ ]:
from typing import Optional


class MLPNorm(nn.Module):
    def __init__(self, norm: Optional[str]):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)
        if norm == "bn":
            self.n1 = nn.BatchNorm1d(128)
        elif norm == "ln":
            self.n1 = nn.LayerNorm(128)
        else:
            self.n1 = None

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        if self.n1 is not None:
            x = self.n1(x)
        x = torch.relu(x)
        return self.fc2(x)


def train_norm(name, norm, epochs=8):
    torch.manual_seed(42)
    model = MLPNorm(norm)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    hist = []
    for _ in range(epochs):
        hist.append(run_epoch(model, train_loader, opt, crit, train=True))
    return hist, name


e_n = 8
h_bn, _ = train_norm("BatchNorm", "bn", e_n)
h_ln, _ = train_norm("LayerNorm", "ln", e_n)
h_none, _ = train_norm("None", None, e_n)

plt.figure(figsize=(7, 4))
plt.plot(range(1, e_n + 1), h_bn, label="BatchNorm")
plt.plot(range(1, e_n + 1), h_ln, label="LayerNorm")
plt.plot(range(1, e_n + 1), h_none, label="No norm")
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.title("MNIST (10k): normalization comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Transformers / LLMs** almost always use **LayerNorm** (or RMSNorm) because sequences have **variable length** and batch statistics are noisier or ill-defined for small or variable batch sizes.

## Section 9: Key Takeaways

- **Backprop** applies the chain rule; frameworks compute the same gradients you derive by hand.
- **Choose the loss** to match the task (MSE vs BCE vs softmax cross-entropy).
- **SGD / Momentum / Adam** trade simplicity, speed, and adaptive scaling — all aim to reduce the loss surface value.
- **Schedules** (warmup, decay) stabilize and refine training.
- **Regularization** (L2, dropout) and **normalization** (Batch vs Layer) change *how* the model generalizes, not just how fast loss drops on the training set.